In [ ]:
# Cell 12: Real-time Inference & Alerting System

import socket
import threading
import queue
import logging
from datetime import datetime
from typing import Dict, List, Any
import smtplib
from email.mime.text import MimeText
from email.mime.multipart import MimeMultipart

class IDSInferenceEngine:
    """Real-time intrusion detection inference engine"""
    
    def __init__(self, model_path: str, meta_path: str, config: Config):
        self.cfg = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Load trained model
        print('[inference] Loading trained model...')
        checkpoint = torch.load(model_path, map_location=self.device)
        self.meta = checkpoint['meta']
        self.label_classes = checkpoint['label_classes']
        
        # Initialize model architecture
        input_dim = self.meta['n_features']
        num_classes = len(self.label_classes)
        
        if config.algo == 'dqn':
            self.model = DQN_MLP(input_dim, num_classes, config.parsed_hidden(), config.dropout)
        else:
            self.model = ActorCritic(input_dim, num_classes, config.parsed_hidden())
        
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.to(self.device)
        self.model.eval()
        
        # Alert thresholds
        self.attack_confidence_threshold = 0.7  # Minimum confidence for attack alert
        self.alert_queue = queue.Queue()
        self.alert_history = []
        
        print(f'[inference] Model loaded successfully!')
        print(f'[inference] Classes: {self.label_classes}')
        print(f'[inference] Input features: {input_dim}')
        
    def preprocess_sample(self, raw_data: Dict[str, Any]) -> torch.Tensor:
        """Convert raw network data to model input format"""
        try:
            # Extract features in the same order as training
            features = []
            feature_names = self.meta['feature_names']
            
            for feat_name in feature_names:
                # Handle different possible field names
                value = raw_data.get(feat_name, 0.0)
                if isinstance(value, str):
                    value = float(value) if value.replace('.', '').replace('-', '').isdigit() else 0.0
                features.append(float(value))
            
            # Convert to numpy and normalize
            features = np.array(features, dtype=np.float32)
            
            # Apply same normalization as training
            means = self.meta['means']
            stds = self.meta['stds']
            features = (features - means) / (stds + 1e-9)
            
            # Convert to tensor
            return torch.from_numpy(features).unsqueeze(0).to(self.device)
            
        except Exception as e:
            print(f'[preprocess] Error: {e}')
            # Return zero tensor as fallback
            return torch.zeros(1, len(self.meta['feature_names'])).to(self.device)
    
    def predict(self, input_tensor: torch.Tensor) -> Dict[str, Any]:
        """Make prediction on preprocessed input"""
        with torch.no_grad():
            if self.cfg.algo == 'dqn':
                logits = self.model(input_tensor)
            else:
                logits, _ = self.model(input_tensor)
            
            # Get probabilities
            probs = torch.softmax(logits, dim=1)
            confidence, predicted_class = torch.max(probs, 1)
            
            predicted_label = self.label_classes[predicted_class.item()]
            confidence_score = confidence.item()
            
            return {
                'predicted_class': predicted_label,
                'confidence': confidence_score,
                'all_probabilities': {
                    self.label_classes[i]: float(probs[0][i]) 
                    for i in range(len(self.label_classes))
                },
                'is_attack': predicted_label.lower() != 'benign',
                'timestamp': datetime.now().isoformat()
            }
    
    def process_network_flow(self, flow_data: Dict[str, Any]) -> Dict[str, Any]:
        """Process a single network flow and return prediction + alert info"""
        # Preprocess
        input_tensor = self.preprocess_sample(flow_data)
        
        # Predict
        result = self.predict(input_tensor)
        
        # Add source information
        result.update({
            'source_ip': flow_data.get('Source_IP', 'unknown'),
            'dest_ip': flow_data.get('Destination_IP', 'unknown'),
            'source_port': flow_data.get('Source_Port', 'unknown'),
            'dest_port': flow_data.get('Destination_Port', 'unknown')
        })
        
        # Check if alert should be triggered
        if (result['is_attack'] and 
            result['confidence'] >= self.attack_confidence_threshold):
            self.trigger_alert(result)
        
        return result
    
    def trigger_alert(self, detection_result: Dict[str, Any]):
        """Trigger security alert"""
        alert = {
            'timestamp': detection_result['timestamp'],
            'alert_type': 'INTRUSION_DETECTED',
            'attack_type': detection_result['predicted_class'],
            'confidence': detection_result['confidence'],
            'source_ip': detection_result['source_ip'],
            'dest_ip': detection_result['dest_ip'],
            'source_port': detection_result['source_port'],
            'dest_port': detection_result['dest_port'],
            'severity': self._get_severity(detection_result['predicted_class'])
        }
        
        # Add to queue for processing
        self.alert_queue.put(alert)
        self.alert_history.append(alert)
        
        # Log immediately
        print(f"🚨 ALERT: {alert['attack_type']} detected from {alert['source_ip']} "
              f"(confidence: {alert['confidence']:.3f})")
    
    def _get_severity(self, attack_type: str) -> str:
        """Determine alert severity based on attack type"""
        high_severity = ['dos', 'ddos', 'infiltration', 'botnet']
        medium_severity = ['portscan', 'bruteforce', 'web attack']
        
        attack_lower = attack_type.lower()
        
        for high in high_severity:
            if high in attack_lower:
                return 'HIGH'
        
        for medium in medium_severity:
            if medium in attack_lower:
                return 'MEDIUM'
        
        return 'LOW'

class AlertManager:
    """Handles alert notifications and logging"""
    
    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.setup_logging()
        
    def setup_logging(self):
        """Setup logging for alerts"""
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler('ids_alerts.log'),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger('IDS_Alerts')
    
    def process_alerts(self, alert_queue: queue.Queue):
        """Process alerts from queue"""
        while True:
            try:
                alert = alert_queue.get(timeout=1)
                self.handle_alert(alert)
                alert_queue.task_done()
            except queue.Empty:
                continue
            except Exception as e:
                self.logger.error(f"Error processing alert: {e}")
    
    def handle_alert(self, alert: Dict[str, Any]):
        """Handle individual alert"""
        # Log to file
        self.logger.warning(
            f"INTRUSION ALERT - Type: {alert['attack_type']}, "
            f"Source: {alert['source_ip']}:{alert['source_port']}, "
            f"Confidence: {alert['confidence']:.3f}, "
            f"Severity: {alert['severity']}"
        )
        
        # Send email notification (if configured)
        if self.config.get('email_alerts', False):
            self.send_email_alert(alert)
        
        # Send to SIEM (if configured)
        if self.config.get('siem_integration', False):
            self.send_to_siem(alert)
    
    def send_email_alert(self, alert: Dict[str, Any]):
        """Send email alert"""
        try:
            msg = MimeMultipart()
            msg['From'] = self.config['email_from']
            msg['To'] = self.config['email_to']
            msg['Subject'] = f"🚨 IDS Alert - {alert['attack_type']} Detected"
            
            body = f"""
            INTRUSION DETECTION ALERT
            
            Attack Type: {alert['attack_type']}
            Confidence: {alert['confidence']:.3f}
            Severity: {alert['severity']}
            
            Source: {alert['source_ip']}:{alert['source_port']}
            Destination: {alert['dest_ip']}:{alert['dest_port']}
            
            Timestamp: {alert['timestamp']}
            
            Please investigate immediately.
            """
            
            msg.attach(MimeText(body, 'plain'))
            
            # Send email (configure your SMTP settings)
            # server = smtplib.SMTP('your-smtp-server.com', 587)
            # server.starttls()
            # server.login(msg['From'], 'your-password')
            # server.send_message(msg)
            # server.quit()
            
            print(f"📧 Email alert sent for {alert['attack_type']}")
            
        except Exception as e:
            self.logger.error(f"Failed to send email alert: {e}")
    
    def send_to_siem(self, alert: Dict[str, Any]):
        """Send alert to SIEM system"""
        # Implement SIEM integration here
        # Example: send to Splunk, ELK stack, etc.
        pass

# Example usage functions
def simulate_network_data():
    """Simulate incoming network flow data"""
    import random
    
    # Simulate realistic network flow features
    return {
        'Flow_Duration': random.uniform(0, 1000000),
        'Total_Fwd_Packets': random.randint(1, 100),
        'Total_Backward_Packets': random.randint(1, 100),
        'Total_Length_of_Fwd_Packets': random.uniform(0, 10000),
        'Total_Length_of_Bwd_Packets': random.uniform(0, 10000),
        'Fwd_Packet_Length_Max': random.uniform(0, 1500),
        'Fwd_Packet_Length_Min': random.uniform(0, 100),
        'Fwd_Packet_Length_Mean': random.uniform(0, 500),
        'Fwd_Packet_Length_Std': random.uniform(0, 200),
        'Bwd_Packet_Length_Max': random.uniform(0, 1500),
        'Source_IP': f"192.168.1.{random.randint(1, 254)}",
        'Destination_IP': f"10.0.0.{random.randint(1, 254)}",
        'Source_Port': random.randint(1024, 65535),
        'Destination_Port': random.choice([80, 443, 22, 23, 21, 25])
    }

def run_real_time_detection():
    """Main function to run real-time detection"""
    
    # Configuration
    alert_config = {
        'email_alerts': True,  # Set to True to enable email alerts
        'email_from': 'ids@yourcompany.com',
        'email_to': 'security@yourcompany.com',
        'siem_integration': False
    }
    
    # Initialize components
    model_path = os.path.join(cfg.ckpt_dir, 'best_model.pth')
    
    if not os.path.exists(model_path):
        print("❌ Model not found! Please train the model first.")
        return
    
    # Load inference engine
    ids_engine = IDSInferenceEngine(model_path, meta_path, cfg)
    alert_manager = AlertManager(alert_config)
    
    # Start alert processing thread
    alert_thread = threading.Thread(
        target=alert_manager.process_alerts, 
        args=(ids_engine.alert_queue,),
        daemon=True
    )
    alert_thread.start()
    
    print("🔍 IDS Real-time Detection Started!")
    print("Press Ctrl+C to stop")
    
    try:
        while True:
            # Simulate incoming network data
            # In real deployment, this would read from:
            # - Network tap/mirror port
            # - Packet capture (pcap)
            # - Network monitoring tool
            # - Log files
            
            flow_data = simulate_network_data()
            
            # Process the flow
            result = ids_engine.process_network_flow(flow_data)
            
            # Print results (in real deployment, you might log these)
            if result['is_attack']:
                print(f"⚠️  Attack detected: {result['predicted_class']} "
                      f"(confidence: {result['confidence']:.3f}) "
                      f"from {result['source_ip']}")
            else:
                print(f"✅ Benign traffic from {result['source_ip']}")
            
            # Wait before processing next flow
            time.sleep(1)  # Adjust based on your needs
            
    except KeyboardInterrupt:
        print("\n🛑 IDS Detection stopped")
        
        # Show summary
        total_alerts = len(ids_engine.alert_history)
        if total_alerts > 0:
            print(f"\n📊 Session Summary:")
            print(f"   Total alerts: {total_alerts}")
            
            # Count by attack type
            attack_counts = {}
            for alert in ids_engine.alert_history:
                attack_type = alert['attack_type']
                attack_counts[attack_type] = attack_counts.get(attack_type, 0) + 1
            
            print(f"   Attack breakdown:")
            for attack_type, count in attack_counts.items():
                print(f"     - {attack_type}: {count}")

print('[inference] 🚨 Real-time IDS inference system loaded!')
print('[inference] Use run_real_time_detection() to start monitoring')

In [ ]:
# Run this after your model is trained
run_real_time_detection()

In [ ]:
# Example: Read from CSV file (batch processing)
def process_csv_file(file_path: str):
    ids_engine = IDSInferenceEngine(model_path, meta_path, cfg)
    
    for chunk in pd.read_csv(file_path, chunksize=1000):
        for _, row in chunk.iterrows():
            flow_data = row.to_dict()
            result = ids_engine.process_network_flow(flow_data)
            # Process result...

# Example: Read from network capture
def process_pcap_file(pcap_path: str):
    # Use scapy or similar to parse network packets
    from scapy.all import rdpcap, IP, TCP
    
    packets = rdpcap(pcap_path)
    for packet in packets:
        if IP in packet and TCP in packet:
            flow_data = extract_features_from_packet(packet)
            result = ids_engine.process_network_flow(flow_data)

In [ ]:
# In the alert_config dictionary:
alert_config = {
    'email_alerts': True,
    'email_from': 'your-ids@company.com',
    'email_to': 'security-team@company.com',
    'siem_integration': True,  # Enable SIEM integration
    'alert_threshold': 0.8,    # Minimum confidence for alerts
}